# Decorators Decoded

When you see `@something` above a function, that's a decorator. Let's understand what they do.

## What Is a Decorator?

A decorator is a function that wraps another function to add behavior.

```python
@decorator
def func():
    pass

# Is exactly the same as:
def func():
    pass
func = decorator(func)
```

In [ ]:
# Simple decorator example
def shout(func):
    """Decorator that makes output uppercase."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

@shout
def greet(name):
    return f"hello, {name}"

print(greet("world"))  # HELLO, WORLD

## Common Built-in Decorators

In [ ]:
# @staticmethod - No self, just a function in a class
# @classmethod - Gets cls instead of self
# @property - Access method like an attribute

class Circle:
    def __init__(self, radius):
        self._radius = radius
    
    @property
    def radius(self):
        """Get radius."""
        return self._radius
    
    @radius.setter
    def radius(self, value):
        """Set radius with validation."""
        if value < 0:
            raise ValueError("Radius must be positive")
        self._radius = value
    
    @property
    def area(self):
        """Computed property."""
        return 3.14159 * self._radius ** 2
    
    @staticmethod
    def is_valid_radius(value):
        """Utility - no self needed."""
        return value >= 0

c = Circle(5)
print(f"Radius: {c.radius}")  # No parentheses! Property
print(f"Area: {c.area}")
c.radius = 10  # Uses setter
print(f"New area: {c.area}")

## `@functools.wraps` - Preserve Function Metadata

In [ ]:
from functools import wraps

def log_calls(func):
    """Decorator that logs function calls."""
    @wraps(func)  # Preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def process_data(data):
    """Process the data."""
    return data * 2

result = process_data([1, 2, 3])
print(f"Result: {result}")
print(f"Function name: {process_data.__name__}")  # Still 'process_data', not 'wrapper'
print(f"Docstring: {process_data.__doc__}")

## `@functools.lru_cache` - Memoization

In [ ]:
from functools import lru_cache
import time

@lru_cache(maxsize=100)
def slow_fibonacci(n):
    """Calculate fibonacci - cached for speed!"""
    if n < 2:
        return n
    return slow_fibonacci(n-1) + slow_fibonacci(n-2)

# Without cache, this would be VERY slow
start = time.time()
result = slow_fibonacci(35)
print(f"fib(35) = {result}")
print(f"Time: {time.time() - start:.4f}s")

# Second call is instant (cached)
start = time.time()
result = slow_fibonacci(35)
print(f"Cached call time: {time.time() - start:.6f}s")

## Decorators with Arguments

In [ ]:
from functools import wraps

def repeat(times):
    """Decorator factory - returns a decorator."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)  # Note the parentheses!
def say_hello(name):
    print(f"Hello, {name}!")

say_hello("World")

## Common Decorators in AI-Generated Code

In [ ]:
# Timing decorator
import time
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"{func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(0.1)
    return "done"

slow_function()

In [ ]:
# Retry decorator
import random
from functools import wraps

def retry(max_attempts=3):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt + 1} failed: {e}")
                    if attempt == max_attempts - 1:
                        raise
        return wrapper
    return decorator

@retry(max_attempts=3)
def flaky_function():
    if random.random() < 0.7:
        raise ValueError("Random failure!")
    return "Success!"

try:
    result = flaky_function()
    print(result)
except ValueError:
    print("All attempts failed")

## Framework Decorators You'll See

```python
# Flask/FastAPI
@app.route("/users")
def get_users():
    ...

# Pytest
@pytest.fixture
def database():
    ...

@pytest.mark.parametrize("input,expected", [...])
def test_something(input, expected):
    ...

# Click (CLI)
@click.command()
@click.option("--name", default="World")
def hello(name):
    ...

# Dataclass
@dataclass
class User:
    name: str
```

## Summary

| Decorator | Purpose |
|-----------|--------|
| `@property` | Method as attribute |
| `@staticmethod` | No self needed |
| `@classmethod` | Gets cls instead of self |
| `@functools.wraps` | Preserve metadata |
| `@functools.lru_cache` | Memoization |
| `@dataclass` | Auto-generate methods |
| `@app.route` | Web framework routing |
| `@pytest.fixture` | Test setup |

## Next Up

Functions that return functions: closures.

Continue to: [Closures](03-closures.ipynb)